# Navegador notebooks

In [8]:
# Celda 1: Imports
from pathlib import Path
import pandas as pd
import json
from IPython.display import display, HTML, Markdown

try:
    import google.colab
    IN_COLAB = True
    PROJECT_ROOT = Path('/content/Estrategias_No_Convencionales')
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path('.')

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print(f"📂 Proyecto: {PROJECT_ROOT}")
print(f" Notebooks: {NOTEBOOKS_DIR}")
print(f"✅ Existe: {NOTEBOOKS_DIR.exists()}")

# Celda 2: Escaneo automático
def escanear_notebooks():
    if not NOTEBOOKS_DIR.exists():
        print(f"ERROR: No existe {NOTEBOOKS_DIR}")
        return []
    
    notebooks_encontrados = []
    
    for bloque_dir in sorted(NOTEBOOKS_DIR.glob('Bloque_*')):
        if not bloque_dir.is_dir():
            continue
        
        bloque_nombre = bloque_dir.name
        
        for ipynb_path in sorted(bloque_dir.glob('*.ipynb')):
            if ipynb_path.name.startswith('.') or '.ipynb_checkpoints' in str(ipynb_path):
                continue
            
            titulo = ipynb_path.stem
            descripcion = ""
            
            try:
                with open(ipynb_path, 'r', encoding='utf-8') as f:
                    nb = json.load(f)
                
                for cell in nb.get('cells', []):
                    if cell.get('cell_type') == 'markdown':
                        source = ''.join(cell.get('source', []))
                        if source.strip():
                            for linea in source.split('\n'):
                                if linea.startswith('#'):
                                    titulo = linea.lstrip('# ').strip()
                                    break
                            lineas = source.split('\n')
                            desc_lineas = []
                            capturando = False
                            for linea in lineas:
                                if linea.startswith('#'):
                                    capturando = True
                                    continue
                                if capturando and linea.strip() and not linea.startswith('**'):
                                    desc_lineas.append(linea.strip())
                                if len(desc_lineas) >= 2:
                                    break
                            descripcion = ' '.join(desc_lineas)[:200]
                            break
            except Exception as e:
                descripcion = f"(Error leyendo: {str(e)[:50]})"
            
            notebooks_encontrados.append({
                'titulo': titulo,
                'archivo': ipynb_path.name,
                'ruta_completa': ipynb_path,
                'ruta_relativa': ipynb_path.relative_to(PROJECT_ROOT),
                'bloque': bloque_nombre,
                'descripcion': descripcion
            })
    
    return notebooks_encontrados

lista_notebooks = escanear_notebooks()
df_notebooks = pd.DataFrame(lista_notebooks)

print(f"✅ Se encontraron {len(df_notebooks)} notebooks organizados por bloques\n")
print("Distribución por bloque:")
for bloque in sorted(df_notebooks['bloque'].unique()):
    count = len(df_notebooks[df_notebooks['bloque'] == bloque])
    print(f"  • {bloque}: {count} notebooks")

# Celda 3: Menú visual interactivo con botones "Abrir en Colab"
def mostrar_notebooks(filtro_bloque=None, filtro_busqueda=None):
    df = df_notebooks.copy()
    
    if filtro_bloque and filtro_bloque != 'Todos':
        df = df[df['bloque'] == filtro_bloque]
    
    if filtro_busqueda:
        mask = (
            df['titulo'].str.contains(filtro_busqueda, case=False, na=False) |
            df['descripcion'].str.contains(filtro_busqueda, case=False, na=False)
        )
        df = df[mask]
    
    html = """
    <style>
        .nb-card {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            margin: 15px 0;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            transition: transform 0.3s;
        }
        .nb-card:hover { transform: translateY(-3px); box-shadow: 0 8px 12px rgba(0,0,0,0.2); }
        .nb-titulo { font-size: 1.3em; font-weight: bold; margin-bottom: 8px; }
        .nb-bloque { font-size: 0.85em; opacity: 0.9; background: rgba(255,255,255,0.2); padding: 3px 10px; border-radius: 12px; display: inline-block; margin-bottom: 10px; }
        .nb-descripcion { font-size: 0.95em; line-height: 1.5; margin-bottom: 15px; }
        .nb-archivo { font-size: 0.8em; color: #e0e0e0; font-family: monospace; margin-bottom: 12px; }
        .btn { display: inline-block; padding: 8px 16px; border-radius: 5px; text-decoration: none; margin-right: 8px; font-weight: bold; font-size: 0.9em; cursor: pointer; border: none; }
        .btn-colab { background: #f9ab00; color: white; }
        .btn-colab:hover { background: #e09800; }
        .btn-ver { background: #4CAF50; color: white; }
        .btn-ver:hover { background: #45a049; }
        h2 { color: #2c3e50; border-bottom: 3px solid #667eea; padding-bottom: 10px; }
    </style>
    """
    
    html += f"<h2>📓 Notebooks del Proyecto ({len(df)} encontrados)</h2>"
    
    if len(df) == 0:
        html += "<p>No se encontraron notebooks con los filtros aplicados.</p>"
    
    # Reemplaza TU_USUARIO por tu usuario real de GitHub
    GITHUB_USER = "akitxu"
    GITHUB_REPO = "Estrategias-No-Convencionales"
    
    for _, row in df.iterrows():
        ruta_rel = str(row['ruta_relativa']).replace(' ', '%20')
        
        # Enlace directo a Colab (abre el notebook en Colab con el repo montado)
        colab_url = f"https://colab.research.google.com/github/{GITHUB_USER}/{GITHUB_REPO}/blob/main/{ruta_rel}"
        
        btn_colab = f'<a href="{colab_url}" target="_blank" class="btn btn-colab"> Abrir en Colab</a>'
        btn_ver = f'<a href="{ruta_rel}" target="_blank" class="btn btn-ver">️ Ver contenido</a>'
        
        html += f"""
        <div class='nb-card'>
            <div class='nb-titulo'>{row['titulo']}</div>
            <div class='nb-bloque'>{row['bloque']}</div>
            <div class='nb-descripcion'>{row['descripcion'] if row['descripcion'] else 'Sin descripción disponible'}</div>
            <div class='nb-archivo'>📄 {row['archivo']}</div>
            {btn_colab}
            {btn_ver}
        </div>
        """
    
    display(HTML(html))

mostrar_notebooks()

# Celda 4: Filtros
print("FILTROS DISPONIBLES")
print("="*70)
print("\nBloques disponibles:")
for i, bloque in enumerate(['Todos'] + list(df_notebooks['bloque'].unique()), 1):
    count = len(df_notebooks[df_notebooks['bloque'] == bloque]) if bloque != 'Todos' else len(df_notebooks)
    print(f"  {i}. {bloque} ({count} notebooks)")

print("\nFunciones disponibles:")
print("  • mostrar_notebooks()")
print("  • mostrar_notebooks(filtro_bloque='Bloque_I_Comportamiento')")
print("  • mostrar_notebooks(filtro_busqueda='Momentum')")

# Celda 5: Tabla comparativa
def tabla_notebooks():
    print("TABLA DE NOTEBOOKS DEL PROYECTO")
    print("="*70)
    df_tabla = df_notebooks[['titulo', 'bloque', 'archivo']].copy()
    df_tabla.columns = ['Título', 'Bloque', 'Archivo']
    display(df_tabla)
    print(f"\nTotal: {len(df_notebooks)} notebooks")
    print(f"Bloques: {df_notebooks['bloque'].nunique()}")

tabla_notebooks()

# Celda 6: Estadísticas
def estadisticas_notebooks():
    print("ESTADÍSTICAS DE NOTEBOOKS")
    print("="*70)
    
    print("Distribución por bloque:")
    for bloque in df_notebooks['bloque'].unique():
        count = len(df_notebooks[df_notebooks['bloque'] == bloque])
        print(f"  • {bloque}: {count} notebooks")
    
    print(f"\nTotal de notebooks: {len(df_notebooks)}")
    print(f"Total de bloques: {df_notebooks['bloque'].nunique()}")
    
    total_celdas = 0
    total_codigo = 0
    total_markdown = 0
    
    for nb_info in lista_notebooks:
        try:
            with open(nb_info['ruta_completa'], 'r', encoding='utf-8') as f:
                nb = json.load(f)
            celdas = nb.get('cells', [])
            total_celdas += len(celdas)
            total_codigo += sum(1 for c in celdas if c.get('cell_type') == 'code')
            total_markdown += sum(1 for c in celdas if c.get('cell_type') == 'markdown')
        except:
            pass
    
    print(f"\nTotal de celdas en todos los notebooks: {total_celdas}")
    print(f"  • Celdas de código: {total_codigo}")
    print(f"  • Celdas de markdown: {total_markdown}")
    print("="*70)

estadisticas_notebooks()

📂 Proyecto: .
 Notebooks: notebooks
✅ Existe: True
✅ Se encontraron 9 notebooks organizados por bloques

Distribución por bloque:
  • Bloque_III_Eventos: 1 notebooks
  • Bloque_II_Cuantitativas: 5 notebooks
  • Bloque_I_Comportamiento: 3 notebooks


FILTROS DISPONIBLES

Bloques disponibles:
  1. Todos (9 notebooks)
  2. Bloque_III_Eventos (1 notebooks)
  3. Bloque_II_Cuantitativas (5 notebooks)
  4. Bloque_I_Comportamiento (3 notebooks)

Funciones disponibles:
  • mostrar_notebooks()
  • mostrar_notebooks(filtro_bloque='Bloque_I_Comportamiento')
  • mostrar_notebooks(filtro_busqueda='Momentum')
TABLA DE NOTEBOOKS DEL PROYECTO


,Título,Bloque,Archivo
0,Capítulo 8: Event-Driven — Merger Arbitrage co...,Bloque_III_Eventos,08_Cap8_Event-Driven.ipynb
1,%% [markdown],Bloque_II_Cuantitativas,05_Cap5_Pairs_Trading.ipynb
2,📊 Capítulo 6: Statistical Arbitrage,Bloque_II_Cuantitativas,06_Cap6_Statistical_Arbitrage.ipynb
3,Verificación de la Estacionariedad del Spread ...,Bloque_II_Cuantitativas,06_Cap6_verificacion_estacionalidad_spread.ipynb
4,📊 Capítulo 7: Algorithmic Trading con IA/ML,Bloque_II_Cuantitativas,07_Cap7_Algotithmic_Trading.ipynb
5,Blog Capítulo 6: Statistical Arbitrage,Bloque_II_Cuantitativas,Blog_Cap6_Statistical_Arbitrage.ipynb
6,📊 Capítulo 2: Contrarian Investing,Bloque_I_Comportamiento,02_Cap2_Contrarian_Investing.ipynb
7,📊 Capítulo 3: Momentum Crash,Bloque_I_Comportamiento,03_Cap3_Momentum_Crash.ipynb
8,📊 Capítulo 4: Fear & Greed Index (FGI),Bloque_I_Comportamiento,04_Cap4_Fear_Greed_Index.ipynb.ipynb



Total: 9 notebooks
Bloques: 3
ESTADÍSTICAS DE NOTEBOOKS
Distribución por bloque:
  • Bloque_III_Eventos: 1 notebooks
  • Bloque_II_Cuantitativas: 5 notebooks
  • Bloque_I_Comportamiento: 3 notebooks

Total de notebooks: 9
Total de bloques: 3

Total de celdas en todos los notebooks: 164
  • Celdas de código: 84
  • Celdas de markdown: 80


# Imports y configuración 

In [9]:
from pathlib import Path
import pandas as pd
import json
from IPython.display import display, HTML, Markdown

# Detectar entorno y configurar rutas
try:
    import google.colab
    IN_COLAB = True
    PROJECT_ROOT = Path('/content/Estrategias-No-Convencionales')
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path('.')

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print(f" Proyecto: {PROJECT_ROOT}")
print(f"📓 Notebooks: {NOTEBOOKS_DIR}")
print(f"✅ Existe: {NOTEBOOKS_DIR.exists()}")

# Debug: Mostrar estructura de carpetas
if NOTEBOOKS_DIR.exists():
    print("\n🔍 Estructura de carpetas encontrada:")
    for item in sorted(NOTEBOOKS_DIR.iterdir()):
        if item.is_dir():
            print(f"   📁 {item.name}/")
            # Mostrar primeros 3 archivos de cada carpeta
            archivos = list(item.glob('*.ipynb'))[:3]
            for arch in archivos:
                print(f"      📄 {arch.name}")
else:
    print(f"\n❌ ERROR: No existe el directorio {NOTEBOOKS_DIR}")
    print("   Verifica que la estructura del proyecto sea correcta.")

 Proyecto: .
📓 Notebooks: notebooks
✅ Existe: True

🔍 Estructura de carpetas encontrada:
   📁 .ipynb_checkpoints/
      📄 18_Cap18_Detector_Ciclos.ipynb-checkpoint.ipynb
      📄 Crisis financieras_4-checkpoint.ipynb
      📄 pruebas-checkpoint.ipynb
   📁 Bloque_III_Eventos/
      📄 08_Cap8_Event-Driven.ipynb
   📁 Bloque_II_Cuantitativas/
      📄 05_Cap5_Pairs_Trading.ipynb
      📄 06_Cap6_Statistical_Arbitrage.ipynb
      📄 06_Cap6_verificacion_estacionalidad_spread.ipynb
   📁 Bloque_IV_Macro/
   📁 Bloque_I_Comportamiento/
      📄 04_Cap4_Fear_Greed_Index.ipynb.ipynb
      📄 02_Cap2_Contrarian_Investing.ipynb
      📄 .ipynb
   📁 Bloque_VII_Alto_Riesgo/
   📁 Bloque_VI_Nicho/
   📁 Bloque_V_Alternativas/


#  Escaneo automático

In [10]:
def escanear_notebooks():
    """
    Escanea las subcarpetas Bloque_* dentro de notebooks/
    Adaptado para Estrategias No Convencionales (33 capítulos, 8 bloques)
    """
    if not NOTEBOOKS_DIR.exists():
        print(f"❌ ERROR: No existe {NOTEBOOKS_DIR}")
        return []
    
    notebooks_encontrados = []
    
    # Buscar SOLO dentro de carpetas Bloque_*
    bloques_encontrados = sorted([d for d in NOTEBOOKS_DIR.iterdir() 
                                   if d.is_dir() and d.name.startswith('Bloque_')])
    
    if not bloques_encontrados:
        print(f"⚠️  No se encontraron carpetas 'Bloque_*' en {NOTEBOOKS_DIR}")
        print("   Estructura esperada: notebooks/Bloque_I_Comportamiento/, notebooks/Bloque_II_Cuantitativas/, etc.")
        return []
    
    print(f"🔍 Encontradas {len(bloques_encontrados)} carpetas de bloques:\n")
    
    for bloque_dir in bloques_encontrados:
        bloque_nombre = bloque_dir.name
        print(f"   📁 {bloque_nombre}")
        
        for ipynb_path in sorted(bloque_dir.glob('*.ipynb')):
            # Excluir checkpoints y archivos temporales
            if ipynb_path.name.startswith('.') or ipynb_path.name.startswith('~'):
                continue
            if '.ipynb_checkpoints' in str(ipynb_path):
                continue
            
            # Extraer título y descripción
            titulo = ipynb_path.stem
            descripcion = ""
            
            try:
                with open(ipynb_path, 'r', encoding='utf-8') as f:
                    nb = json.load(f)
                
                for cell in nb.get('cells', []):
                    if cell.get('cell_type') == 'markdown':
                        source = ''.join(cell.get('source', []))
                        if source.strip():
                            # Extraer título (primera línea con #)
                            for linea in source.split('\n'):
                                if linea.startswith('#'):
                                    titulo = linea.lstrip('# ').strip()
                                    break
                            # Extraer descripción (primer párrafo tras el título)
                            lineas = source.split('\n')
                            desc_lineas = []
                            capturando = False
                            for linea in lineas:
                                if linea.startswith('#'):
                                    capturando = True
                                    continue
                                if capturando and linea.strip() and not linea.startswith('**'):
                                    desc_lineas.append(linea.strip())
                                if len(desc_lineas) >= 2:
                                    break
                            descripcion = ' '.join(desc_lineas)[:200]
                            break
            except Exception as e:
                descripcion = f"(Error leyendo: {str(e)[:50]})"
            
            notebooks_encontrados.append({
                'titulo': titulo,
                'archivo': ipynb_path.name,
                'ruta_completa': ipynb_path,
                'ruta_relativa': ipynb_path.relative_to(PROJECT_ROOT),
                'bloque': bloque_nombre,
                'descripcion': descripcion
            })
            
            print(f"      ✅ {ipynb_path.name}")
    
    return notebooks_encontrados

# Ejecutar escaneo
lista_notebooks = escanear_notebooks()

if lista_notebooks:
    df_notebooks = pd.DataFrame(lista_notebooks)
    print(f"\n✅ Se encontraron {len(df_notebooks)} notebooks organizados por bloques\n")
    print("Distribución por bloque:")
    for bloque in sorted(df_notebooks['bloque'].unique()):
        count = len(df_notebooks[df_notebooks['bloque'] == bloque])
        print(f"  • {bloque}: {count} notebooks")
else:
    print("\n❌ No se encontraron notebooks. Verifica la estructura del proyecto.")
    df_notebooks = pd.DataFrame()  # DataFrame vacío para evitar errores posteriores

🔍 Encontradas 7 carpetas de bloques:

   📁 Bloque_III_Eventos
      ✅ 08_Cap8_Event-Driven.ipynb
   📁 Bloque_II_Cuantitativas
      ✅ 05_Cap5_Pairs_Trading.ipynb
      ✅ 06_Cap6_Statistical_Arbitrage.ipynb
      ✅ 06_Cap6_verificacion_estacionalidad_spread.ipynb
      ✅ 07_Cap7_Algotithmic_Trading.ipynb
      ✅ Blog_Cap6_Statistical_Arbitrage.ipynb
   📁 Bloque_IV_Macro
   📁 Bloque_I_Comportamiento
      ✅ 02_Cap2_Contrarian_Investing.ipynb
      ✅ 03_Cap3_Momentum_Crash.ipynb
      ✅ 04_Cap4_Fear_Greed_Index.ipynb.ipynb
   📁 Bloque_VII_Alto_Riesgo
   📁 Bloque_VI_Nicho
   📁 Bloque_V_Alternativas

✅ Se encontraron 9 notebooks organizados por bloques

Distribución por bloque:
  • Bloque_III_Eventos: 1 notebooks
  • Bloque_II_Cuantitativas: 5 notebooks
  • Bloque_I_Comportamiento: 3 notebooks


# Menú visual interactivo

In [11]:
def mostrar_notebooks(filtro_bloque=None, filtro_busqueda=None):
    """Muestra los notebooks con enlaces funcionales"""
    
    if df_notebooks.empty:
        print("⚠️  No hay notebooks disponibles para mostrar.")
        print("   Ejecuta primero la celda de escaneo (Celda 2).")
        return
    
    df = df_notebooks.copy()
    
    # Aplicar filtros
    if filtro_bloque and filtro_bloque != 'Todos':
        df = df[df['bloque'] == filtro_bloque]
    
    if filtro_busqueda:
        mask = (
            df['titulo'].str.contains(filtro_busqueda, case=False, na=False) |
            df['descripcion'].str.contains(filtro_busqueda, case=False, na=False)
        )
        df = df[mask]
    
    html = """
    <style>
        .nb-card {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            margin: 15px 0;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            transition: transform 0.3s;
        }
        .nb-card:hover { transform: translateY(-3px); box-shadow: 0 8px 12px rgba(0,0,0,0.2); }
        .nb-titulo { font-size: 1.3em; font-weight: bold; margin-bottom: 8px; }
        .nb-bloque {
            font-size: 0.85em; opacity: 0.9;
            background: rgba(255,255,255,0.2);
            padding: 3px 10px; border-radius: 12px;
            display: inline-block; margin-bottom: 10px;
        }
        .nb-descripcion { font-size: 0.95em; line-height: 1.5; margin-bottom: 15px; }
        .nb-archivo { font-size: 0.8em; color: #e0e0e0; font-family: monospace; margin-bottom: 12px; }
        .btn {
            display: inline-block; padding: 8px 16px; border-radius: 5px;
            text-decoration: none; margin-right: 8px; font-weight: bold;
            font-size: 0.9em; cursor: pointer; border: none;
        }
        .btn-colab { background: #f9ab00; color: white; }
        .btn-colab:hover { background: #e09800; }
        .btn-ver { background: #4CAF50; color: white; }
        .btn-ver:hover { background: #45a049; }
        h2 { color: #2c3e50; border-bottom: 3px solid #667eea; padding-bottom: 10px; }
    </style>
    """
    
    html += f"<h2>📓 Notebooks del Proyecto ({len(df)} encontrados)</h2>"
    
    if len(df) == 0:
        html += "<p>No se encontraron notebooks con los filtros aplicados.</p>"
    
    # Reemplaza TU_USUARIO por tu usuario real de GitHub
    GITHUB_USER = "akitxu"
    GITHUB_REPO = "Estrategias-No-Convencionales"
    
    for _, row in df.iterrows():
        ruta_rel = str(row['ruta_relativa']).replace(' ', '%20')
        
        # Enlace directo a Colab
        colab_url = f"https://colab.research.google.com/github/{GITHUB_USER}/{GITHUB_REPO}/blob/main/{ruta_rel}"
        
        btn_colab = f'<a href="{colab_url}" target="_blank" class="btn btn-colab">🚀 Abrir en Colab</a>'
        btn_ver = f'<a href="{ruta_rel}" target="_blank" class="btn btn-ver">📄 Ver contenido</a>'
        
        html += f"""
        <div class='nb-card'>
            <div class='nb-titulo'>{row['titulo']}</div>
            <div class='nb-bloque'>{row['bloque']}</div>
            <div class='nb-descripcion'>{row['descripcion'] if row['descripcion'] else 'Sin descripción disponible'}</div>
            <div class='nb-archivo'>📄 {row['archivo']}</div>
            {btn_colab}
            {btn_ver}
        </div>
        """
    
    display(HTML(html))

# Mostrar todos los notebooks
mostrar_notebooks()

# Diagnóstico rápido

In [7]:
print("🔍 DIAGNÓSTICO DE ESTRUCTURA")
print("="*70)
print(f"\n📂 PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   Existe: {PROJECT_ROOT.exists()}")

if PROJECT_ROOT.exists():
    print(f"\n Contenido de PROJECT_ROOT:")
    for item in sorted(PROJECT_ROOT.iterdir()):
        tipo = "📁" if item.is_dir() else "📄"
        print(f"   {tipo} {item.name}")
    
    if NOTEBOOKS_DIR.exists():
        print(f"\n📁 Contenido de notebooks/:")
        for item in sorted(NOTEBOOKS_DIR.iterdir()):
            if item.is_dir():
                print(f"   📁 {item.name}/")
                archivos = list(item.glob('*.ipynb'))
                print(f"      ({len(archivos)} notebooks)")
            else:
                print(f"   📄 {item.name}")
    else:
        print(f"\n❌ No existe la carpeta notebooks/")
        print("   Estructura esperada:")
        print("   Estrategias-No-Convencionales/")
        print("   └── notebooks/")
        print("       ├── Bloque_I_Comportamiento/")
        print("       ├── Bloque_II_Cuantitativas/")
        print("       └── ...")

🔍 DIAGNÓSTICO DE ESTRUCTURA

📂 PROJECT_ROOT: .
   Existe: True

 Contenido de PROJECT_ROOT:
   📁 .git
   📄 .gitignore
   📁 .ipynb_checkpoints
   📁 .virtual_documents
   📄 00_Navegador_Capitulos.ipynb
   📄 00_Navegador_Notebooks.ipynb
   📄 LICENSE
   📄 NOTA_LEGAL.md
   📄 README.md
   📁 carteras
   📁 chapters
   📁 core
   📁 datos
   📄 disclaimer.py
   📁 logs
   📁 notebooks
   📁 outputs
   📄 renombrar_carpetas.ipynb

📁 Contenido de notebooks/:
   📁 .ipynb_checkpoints/
      (30 notebooks)
   📄 00_Indice.pdf
   📄 00_Prefacio_y_Arquitectura.ipynb
   📄 00_Prefacio_y_Arquitectura.py
   📄 01_patrones_armonicos.ipynb.ipynb
   📁 Bloque_III_Eventos/
      (1 notebooks)
   📁 Bloque_II_Cuantitativas/
      (5 notebooks)
   📁 Bloque_IV_Macro/
      (0 notebooks)
   📁 Bloque_I_Comportamiento/
      (4 notebooks)
   📁 Bloque_VII_Alto_Riesgo/
      (0 notebooks)
   📁 Bloque_VI_Nicho/
      (0 notebooks)
   📁 Bloque_V_Alternativas/
      (0 notebooks)
